# AML Report 

### Imports

In [0]:
from pyspark.sql.functions import row_number, count, max
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DecimalType
from datetime import datetime
from random import randint
from pyspark.sql.functions import crc32, concat_ws
from io import BytesIO
import pandas as pd
import os
from pyspark.sql.window import Window
from pyspark.sql.functions import monotonically_increasing_id


In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError

from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()
print(env)

## Insert existing AML Table and Calendar

In [0]:
aml_report_view = spark.sql(f""" SELECT * FROM `{env}-cdh`.delivery.aml_report""")
aml_report_view.createOrReplaceTempView("aml_report_view")

execution_calendar = spark.sql(f""" SELECT * FROM `{env}-cdh`.delivery.execution_calendar""")
execution_calendar.createOrReplaceTempView("execution_calendar")


In [0]:
%python

## Generate new aml_sch_id (random 4 digit number) that does not already exist in aml_report
existing_ids = [row['aml_sch_id'] for row in spark.sql("SELECT DISTINCT aml_sch_id FROM aml_report_view").collect()]

while True:
    new_aml_sch_id = randint(1000, 9999)
    if new_aml_sch_id not in existing_ids:
        break

### Businessdate

In [0]:
Businessdate = spark.sql(f"""                    
                        select PreviousBusinessDay
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate =cast(getdate() as date)
                    """).first()[0]

print(Businessdate)

### Check if AML can run for the current month

In [0]:
InitialCalendar = spark.sql(f"""
    SELECT DISTINCT a.id,
        CalendarYear AS ExecYear,  
        'M' AS ExecType,
        CalendarMonth AS ExecPart,
        FirstDateOfMonth AS DateFrom,
        LastDateOfMonth AS DateTo,
        PreviousMonthLastDate,
        CAST(CONCAT(CalendarYear, '-01-01') AS DATE) AS AMLYear,
        {new_aml_sch_id} AS aml_sch_id,
        a.ActualExecution,
        a.ExecPart,
        a.ExecType, 
        a.ExecutionDate
    FROM `{env}-cdh`.reference.ref_calendar i
    JOIN execution_calendar a
    ON i.FirstDateOfMonth = a.DateFrom
    WHERE CalendarYear = YEAR(CURRENT_DATE())
     -- AND IsBusinessDay = 1
      AND MONTH(CalendarDate) = MONTH(DATE(getdate())) - 1
     

""")

InitialCalendar.createOrReplaceTempView("InitialCalendar")

AmlExecutionCalendar = spark.sql(f""" 
                                 SELECT * 
                                 FROM InitialCalendar
                                 WHERE (ActualExecution IS NULL OR ActualExecution  == "NULL")
                                 AND ExecutionDate < cast(getdate() as date)
                                 """)
AmlExecutionCalendar.createOrReplaceTempView("AmlExecutionCalendar")


# If df is not empty then we can execute the AML Process
if AmlExecutionCalendar.count() == 0:
    AMLExecutionFlag = 0
else:
    AMLExecutionFlag = 1

print(AMLExecutionFlag)

if AMLExecutionFlag == 1 :
    print("AML can run for", AmlExecutionCalendar.collect()[0]['DateFrom'], AmlExecutionCalendar.collect()[0]['DateTo'])

    ## DateFrom
    DateFrom = spark.sql(f"""                    
                        select datefrom from AmlExecutionCalendar
                       
                    """).first()[0]

    print(DateFrom)

    ## DateTo
    DateTo = spark.sql(f"""                    
                        select dateto from AmlExecutionCalendar
                       
                    """).first()[0]

    print(DateTo)

    ## AMLYear
    AMLYear = spark.sql(f"""                    
                        select AMLYear from AmlExecutionCalendar
                       
                    """).first()[0]

    print(AMLYear)


    ## PreviousMonthDateTo
    PreviousMonthDateTo = spark.sql(f"""                    
                        select PreviousMonthLastDate from AmlExecutionCalendar
                       
                    """).first()[0]

    print(PreviousMonthDateTo)

    ## ExecutionDare
    ExecutionDate = spark.sql(f"""                    
                        select ExecutionDate from AmlExecutionCalendar
                         """).first()[0]
    print(ExecutionDate)




## Create AmlPerimeter

### DExposure

In [0]:
if AMLExecutionFlag == 1 :
    
    DExposure = spark.sql(f"""        
                            SELECT  DISTINCT act.exposurekey,
                                             l.CDI, 
                                             act.BusinessDate
                                    FROM  `{env}-cdh`.derivativeactive.Dexposure l  
                                    join primary.accounttransaction act
                                    on act.exposurekey = l.exposurekey
                                    WHERE act.BusinessDate = '{Businessdate}'
                                                                   """)

    DExposure.createOrReplaceTempView("DExposure")

### DParty

In [0]:
if AMLExecutionFlag == 1:
        DParty = spark.sql(f"""        
                            SELECT  prt.cdi
                                    , prt.PartyType               PartyType
                                    , prt.TaxRegistrationNumber TaxNumber
                                    , IFNULL(prt.FirstNameGR,'') FirstName
                                    , IFNULL(prt.LastNameGR,'') LastName
                                    , IFNULL(prt.FirstFatherNameGR,'') FatherName
                                    , IFNULL(prt.Nationality,'') Nationality
                                    , prt.BirthDate
                                    , IFNULL(prt.Occupation,'') Occupation
                                    , prt.PortfolioName
                                    , CASE WHEN PartyType = 'Organization' THEN 'Yes' ELSE 'No' END AS LegalEntity
                                    FROM `{env}-cdh`.derivative.Dparty prt  
                                    WHERE prt.BusinessDate = '{Businessdate}' 
                                                                   """)

        DParty.createOrReplaceTempView("DParty")

### Collection

In [0]:
if AMLExecutionFlag == 1:
        Collection = spark.sql(f"""        
                            SELECT  DISTINCT CollectionKey
                                    , cdi
                                    , exposurekey
                                    , loanbankcode
                                    , cast(ActualDate as date) PayDate
                                    , cast(ActualDate as date) PayDateRd
                                    , Amount                   PaymentAmount
                                    , PortfolioName
                                    , if(ProfitsPaymentType ='N/A','cash',ProfitsPaymentType) PaymentType 
                                    , PaymentTypeGroup
                                    , PortfolioKey
                                    , LoanBankCode
                                    , BusinessDate
                                    FROM `{env}-cdh`.primaryactive.collection  
                                    WHERE FinalReason = 'Matched'
                                    AND  (ltrim(rtrim(PaymentTypeGroup)) not in (
                                                        'RE Collateral Liquidation - Repossession'
                                                        ,'Government Subsidy'
                                                        ,'RE Collateral Liquidation - 3rd Party'
                                                        ) 
                                            OR PaymentTypeGroup is null)
                                    AND (ltrim(rtrim(ProfitsPaymentType))
                                    not in (SELECT Description FROM `{env}-cdh`.reference.paymenttype_auctions WHERE FieldName = 'PaymentType')  
                                    OR ProfitsPaymentType is null)
                                    AND ltrim(rtrim(ProfitsPaymentType)) not like '%Collection through%' 
                                    AND ltrim(rtrim(ProfitsPaymentType)) not like '%Gross Proceeds%' 
                                    AND ltrim(rtrim(ProfitsPaymentType)) not like '%_REO' 
                                    AND ltrim(rtrim(ProfitsPaymentType)) not like '%Non-RE Collateral Auction Proceeds%' 
                                    AND BusinessDate = '{Businessdate}'     
                                """)       

        Collection.createOrReplaceTempView("Collection")

### Final Perimeter

In [0]:
if AMLExecutionFlag ==1:
        amlPerimeter = spark.sql(f"""        
                            SELECT  DISTINCT prt.CDI
                                    , l.ExposureKey
                                    , coll.PayDate
                                    , coll.PayDateRd
                                    , coll.PaymentAmount
                                    , prt.PartyType
                                    , coll.PortfolioName 
                                    , prt.PortfolioName CustomerPortfolio
                                    , coll.PaymentType 
                                    , prt.TaxNumber
                                    , prt.FirstName
                                    , prt.LastName
                                    , prt.FatherName
                                    , prt.Nationality
                                    , prt.BirthDate
                                    , prt.Occupation
                                    , prt.LegalEntity
                                    , coll.CollectionKey
                                    FROM  Collection coll 
                                    JOIN `{env}-cdh`.primaryactive.portfolio por
                                    ON por.unit_code_profits=coll.portfolioKey         
                                    JOIN DExposure l    
                                    ON l.ExposureKey = CASE WHEN LEFT(coll.loanbankcode,3)= por.Prefix THEN coll.loanbankcode
                                    ELSE concat(por.Prefix,coll.loanbankcode) END
                                    JOIN DParty prt  
                                    ON prt.cdi = l.cdi
                                    JOIN AMLExecutionCalendar ex on coll.PayDateRD between AMLYear AND DateTo
                                    WHERE prt.TaxNumber is not NULL""")

        amlPerimeter.createOrReplaceTempView("amlPerimeter")


### Portfolios Perimeter

In [0]:
## make case when dynamic if its in use 
PortfolioPerimeter = spark.sql(f"""                    
                            SELECT distinct p.portfolioname, Prefix
                            FROM `{env}-cdh`.primary.portfolio p
                            WHERE  unit_code_profits IS NOT NULL 
                            AND p.PortfolioName NOT LIKE '%common%' 
                            AND p.PortfolioName NOT LIKE 'N/A' 
                            AND p.IsBankingFlag = 'FALSE'

                        """)

PortfolioPerimeter.createOrReplaceTempView("PortfolioPerimeter")

### Perimeter of all customers with multiple Portfolios

In [0]:
if AMLExecutionFlag==1:
     cc = spark.sql(f"""
                select distinct  p.TaxRegistrationNumber TaxNumber, pp.Prefix portfolio
                , pp.PortfolioName origin
                FROM `{env}-cdh`.derivativeactive.DParty p
                join`{env}-cdh`.primaryactive.exposure e
                on p.cdi = e.cdi
                join PortfolioPerimeter pp on pp.PortfolioName = p.PortfolioName
                WHERE PartyType in ('Guarantor','Primary Owner','Other') """)

     cc.createOrReplaceTempView("cc")

     ## All Customers from the current perimeter with at least 2 portfolios per TaxNumber
     amlCommonCustomersDuplicates = spark.sql(f""" SELECT cc.TaxNumber,  COUNT(DISTINCT cc.Origin)
                                    FROM amlPerimeter a
                                    join cc on a.TaxNumber = cc.TaxNumber
                                    WHERE a.PortfolioName in (select PortfolioName from PortfolioPerimeter)
                                    GROUP BY cc.TaxNumber
                                    HAVING COUNT(DISTINCT a.PortfolioName)>1
                               """)

     amlCommonCustomersDuplicates.createOrReplaceTempView("amlCommonCustomersDuplicates")


     ## All Customers from the current perimeter with at least 2 portfolios per TaxNumber
     amlCommonCustomers = spark.sql(f""" SELECT DISTINCT cc.TaxNumber, cc.Origin
                                    FROM amlCommonCustomersDuplicates a
                                    join cc on a.TaxNumber = cc.TaxNumber
                                    WHERE cc.Origin in (select PortfolioName from PortfolioPerimeter)
                                    
                               """)

     amlCommonCustomers.createOrReplaceTempView("amlCommonCustomers")

## Create aml_sch_id 

## Generate Calendar and Check if AML can run for the current month

# Functions

### Find Tax Numbers with duplicate Payment Types

In [0]:
def find_duplicate_payment_type(CustomersPerimeter, datefrom: str, dateto: str, view_name: str, spark):
    CustomersPerimeter.createOrReplaceTempView("CustomersPerimeterInput")

    # Run the SQL to create the DuplicatePaymentFlags view
    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT a.TaxNumber,
               COUNT(DISTINCT a.paymenttype) AS countPaymentType
        FROM amlPerimeter a
        join CustomersPerimeterInput b on a.Taxnumber = b.Taxnumber
        WHERE a.PortfolioName IN (SELECT c.PortfolioName FROM PortfolioPerimeter c)
          AND a.PayDateRd BETWEEN DATE('{datefrom}') AND DATE('{dateto}')
          AND year(a.PayDateRd) = year(getdate())
        GROUP BY a.TaxNumber
        HAVING COUNT(DISTINCT a.paymenttype) > 1


    """
    spark.sql(query)


### Calculate min and max Pay Date per Tax Number

In [0]:
def calculate_min_max_paydate(CustomersPerimeter,view_name: str, spark):
    CustomersPerimeter.createOrReplaceTempView("CustomersPerimeterInput")

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT DISTINCT Ruleid,  
								b.taxnumber, 
								b.PaymentAmount, 
								NumOfPayment, 
								NumOfPortfolio, 
								mindate.FirstDateOfMonth minPaydate, 
								maxdate.LastDateOfMonth maxpaydate, 
								c.PortfolioName
								FROM CustomersPerimeterInput b
								join amlPerimeter c on b.taxnumber = c.TaxNumber  
								join `{env}-cdh`.reference.ref_calendar mindate
								on b.minPayDate = mindate.CalendarDate
								join `{env}-cdh`.reference.ref_calendar maxdate
								on maxPayDate = maxdate.CalendarDate
								where c.PortfolioName in (select PortfolioName from PortfolioPerimeter)  
								
								
    """
    spark.sql(query)



### Calculate the Ytd Payments for specific time periods

In [0]:
from datetime import date
def calculate_YtD_Payments(datefrom: date, dateto: date, view_name: str, spark):

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
                                            SELECT   TaxNumber
		  											, sum(PaymentAmount) SumPaymentAmountYtd
                                                    , count(1) NumOfPayment
                                                    , count(distinct PortfolioName) NumOfPortfolio
											from amlPerimeter
											where PortfolioName in (select PortfolioName from PortfolioPerimeter) 
												--and TaxNumber in  (SELECT TaxNumber FROM ExistingCustomersCurrentMonth)
                                                and PayDateRd   between  DATE('{datefrom.isoformat()}') AND DATE('{dateto.isoformat()}')
											group by TaxNumber
											HAVING sum(PaymentAmount) >= 15000
								
								
    """
    spark.sql(query)


 

## Aggregation of Portfolios (Multiple Portfolio per TaxNumber)

In [0]:
def update_portfolio_of_payment(datefrom: str, dateto: str, spark):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW CustomerPortfolioAggtemp AS
        SELECT a.taxnumber,CONCAT_WS(',', COLLECT_LIST(a.PortfolioName)) AS portfolio
        FROM (
            SELECT DISTINCT a.taxnumber, a.PortfolioName
            FROM amlPerimeter a
            JOIN (
                SELECT taxnumber, PayDateRd
                FROM amlPerimeter
                GROUP BY taxnumber, PayDateRd
            ) mdate ON mdate.taxnumber = a.taxnumber
                  AND mdate.PayDateRd = a.PayDateRd
            INNER JOIN amlReportRun r ON a.taxnumber = r.Tax_number
            WHERE a.PayDateRd BETWEEN DATE('{datefrom}') AND DATE('{dateto}')
        ) a
        GROUP BY a.taxnumber
    """)

## Flag for January Run

In [0]:

if AMLExecutionFlag==1:
    ## Check if we are currently in the January Run or not
    JanuaryFlag = spark.sql(f"""                    
                        select case when month(ExecutionDate)-1 = 1 then 1 else 0 end JanuaryFlag from AMLExecutionCalendar
                       
                    """).first()[0]

    print(JanuaryFlag)

### January Rule Calculations

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 1:
  
    # Existing Customers
    calculate_YtD_Payments(DateFrom, DateTo, "JanuaryCustomers_df", spark)
    JanuaryCustomers= spark.sql("SELECT * FROM JanuaryCustomers_df")
    JanuaryCustomers.createOrReplaceTempView("JanuaryCustomers")

    ## Current M Ytd: Calculate for all the current Customers the Year to Date Sum of all payments 
    ## Calculation Period: Start of Year till End of Calculation Period (End of Calculation Month)
      

    find_duplicate_payment_type(JanuaryCustomers,DateFrom, DateTo, "DuplicatePaymentFlagsJanuary", spark)

    DuplicatePaymentTypeJanuary = spark.sql("SELECT DISTINCT TaxNumber FROM DuplicatePaymentFlagsJanuary")
    DuplicatePaymentTypeJanuary.createOrReplaceTempView("DuplicatePaymentTypeJanuary")

    FinalRule23 = spark.sql(f"""                    
                       
                              SELECT	DISTINCT  RuleId
                            , a.TaxNumber
                            , a.PaymentAmount
                            , NumOfPayment
                            , NumOfPortfolio
                            , CASE WHEN dp.TaxNumber THEN b.PaymentType ELSE NULL END AS PaymentType
                            , b.PortfolioName
                        FROM JanuaryCustomers a
                        JOIN amlPerimeter	b
                        ON a.TaxNumber = b.TaxNumber
                        --JOIN AmlExecutionCheck 
                        --ON PayDateRd between DateFrom and DateTo    
                        LEFT JOIN DuplicatePaymentTypeJanuary dp
                        ON dp.TaxNumber = a.TaxNumber  
                        WHERE PayDateRd between DateFrom and DateTo                    
                                          """)
    FinalRule23.createOrReplaceTempView("FinalRule23")


# Other Months (Except for January)

### Create Initial Perimeters (Existing and New Customers)

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    

	## Find customers that already exist in previous Months in the final AML table  
	ExistInPrevMonths = spark.sql(f"""
								SELECT DISTINCT AmlExecutionCalendar.id, Tax_Number, account_number, payment_amount
                                FROM aml_report_view	
                                join AmlExecutionCalendar on date_from between AMLYear AND PreviousMonthLastDate	
								""")	

	ExistInPrevMonths.createOrReplaceTempView("ExistInPrevMonths")
 
	## Fine new customers (no extra calculation rules - just need to be over 15k)
	NewCustomers = spark.sql(f"""
								SELECT   23 RuleId
										, TaxNumber 
										, sum  (PaymentAmount) PaymentAmount
										, count(1) NumOfPayment
										, count(distinct PortfolioName) NumOfPortfolio 
										, min(paydaterd) minPayDate  
										, max(paydaterd) maxPayDate 
										from amlPerimeter
										join AmlExecutionCalendar on PayDateRd   between AMLYear and DateTo  
										where PortfolioName in (select PortfolioName from PortfolioPerimeter)  
											  and TaxNumber not in  (SELECT Tax_Number FROM ExistInPrevMonths)
										group by TaxNumber
										HAVING sum(PaymentAmount) >= 15000	
								""")	

	NewCustomers.createOrReplaceTempView("NewCustomers")

	## Find Customers that have made transactions during the current calculation month
	ExistingCustomersCurrentMonth = spark.sql(f""" 
												SELECT  DISTINCT TaxNumber
												from amlPerimeter
												join AmlExecutionCalendar on PayDateRd   between DateFrom and DateTo  
												where PortfolioName in (select PortfolioName from PortfolioPerimeter)  
											    and TaxNumber in  (SELECT Tax_Number FROM ExistInPrevMonths)
												group by TaxNumber 
												HAVING sum(PaymentAmount) >= 15000
            								""")
	ExistingCustomersCurrentMonth.createOrReplaceTempView("ExistingCustomersCurrentMonth")
	




### Current MtD

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Current M Ytd: Calculate for all the current Customers the Year to Date Sum of all payments 
    ## Calculation Period: Start of Year till End of Calculation Period (End of Calculation Month)

    # Existing Customers
    calculate_YtD_Payments(AMLYear, DateTo, "CurrentMYtd", spark)
    ExistingCustomersYtd= spark.sql("SELECT * FROM CurrentMYtd WHERE TaxNumber in  (SELECT TaxNumber FROM ExistingCustomersCurrentMonth) ")
    ExistingCustomersYtd.createOrReplaceTempView("ExistingCustomersYtd")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Previous M Ytd: Calculate for all the current Customers the Year to Date Sum of all payments 
    ## Calculation Period: Start of Year till End of Previous Month
    calculate_YtD_Payments(AMLYear, PreviousMonthDateTo, "PreviousMYtd", spark)
    PreviousMCustomersYtd= spark.sql("SELECT * FROM PreviousMYtd WHERE TaxNumber in  (SELECT TaxNumber FROM ExistingCustomersCurrentMonth)")
    PreviousMCustomersYtd.createOrReplaceTempView("PreviousMCustomersYtd")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
## Create conditions for each hit/match category according to AML specs for Rule 23 */
	## If the sum was <= 15k -> Report it the first time its over 15k
	## If its above 15k then only report it if we enter the next hit/match category and the delta is 50k CurrentMonthYtd Sum 
	## e.g. PreviousYtd = 180.000, New Payments = 15.000 -> CurrentYtd = 195.000
	## The next hit/match is (180k/50k + 1) * 50k  = 200k but the CurrentYtf is 195k --> do not report it!
	AlertConditions = spark.sql(f"""
										SELECT
											currentytd.Taxnumber,
											previousytd.SumPaymentAmountYtd PreviousYtd,
											currentytd.SumPaymentAmountYtd CurrentYtd,

											CASE 
												WHEN previousytd.SumPaymentAmountYtd <15000 AND currentytd.SumPaymentAmountYtd >=15000 THEN 1
												WHEN (currentytd.SumPaymentAmountYtd) >= cast(((previousytd.SumPaymentAmountYtd/50000) + 1) as int) * 50000 THEN 1 
											ELSE 0 END AS ReportFlag
										FROM ExistingCustomersYtd currentytd
										join PreviousMCustomersYtd previousytd
										ON currentytd.TaxNumber = previousytd.TaxNumber 
										GROUP BY currentytd.Taxnumber, previousytd.SumPaymentAmountYtd,	currentytd.SumPaymentAmountYtd 
									""")
	AlertConditions.createOrReplaceTempView("AlertConditions")
	
	## New Transactions: Find all customers in current month that satisfy the alert conditions
	## and that have not already been inserted to the final AML table
	ExistingCustomersNewTransactionsInserted = spark.sql(f"""	
											select   
												perim.TaxNumber
												, perim.PaymentAmount
												, PortfolioName
												,PayDateRd
											from amlPerimeter perim
											join AmlExecutionCalendar on PayDateRd   between AMLYear and DateTo  
											join AlertConditions a on a.Taxnumber = perim.TaxNumber
											where PortfolioName in (select portfolioname from PortfolioPerimeter)  
											and a.ReportFlag =1
											AND year(paydaterd) = year(AMLYear)
			""")
	ExistingCustomersNewTransactionsInserted.createOrReplaceTempView("ExistingCustomersNewTransactionsInserted")

	## Existing/to be excuded Transactions: All customers existing in the aml table before the current run
	ExistingCustomersExcluded = spark.sql(f"""
												SELECT   aml.Tax_Number
													, perim.PaymentAmount
													, portfolio_payment
													, PayDateRd
												FROM aml_report_view aml
												join amlPerimeter perim on aml.Tax_Number = perim.TaxNumber 
												AND PayDateRd  between aml.date_from and aml.date_to
												AND Year(PayDateRd) = Year(getdate())
												
												""")	

	ExistingCustomersExcluded.createOrReplaceTempView("ExistingCustomersExcluded")	

	## Find all distinct Payments that have not been inserted in the AML table prior to this run	
	ExistingCustomersSum = spark.sql(f"""
										SELECT 
											23 AS RuleId, 
											taxnumber,
											COUNT(1) AS NumOfPayment, 
											COUNT(DISTINCT PortfolioName) AS NumOfPortfolio,
											SUM(paymentamount) AS PaymentAmount,
											min(paydaterd) minPayDate,
											max(paydaterd) maxPayDate 
										FROM (
											SELECT *
											FROM ExistingCustomersNewTransactionsInserted
											EXCEPT
											SELECT *
											FROM ExistingCustomersExcluded
										) AS r
										WHERE r.PortfolioName IN (SELECT PortfolioName FROM PortfolioPerimeter)
										GROUP BY taxnumber;

									""" )
	ExistingCustomersSum.createOrReplaceTempView("ExistingCustomersSum")
	

### Existing Customers min/max date and duplicate Payment Types calculation

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Calculate the minimum and maximum date for payments per Taxnumber
    ## e.g. If a taxnumber was not inserted in M2 (February), 
    ## and now the sum satisfies the hit/match conditions --> therefore the M period is 01/02-end of current month

    # Existing Customers
    calculate_min_max_paydate(ExistingCustomersSum, "FinalCalculationFieldsExisting",spark)
    FinalExistingCustomers= spark.sql("SELECT * FROM FinalCalculationFieldsExisting")
    FinalExistingCustomers.createOrReplaceTempView("FinalExistingCustomers")

    find_duplicate_payment_type(FinalExistingCustomers, DateFrom,  DateTo,"DuplicatePaymentFlags_Existing", spark)
    DuplicatePaymentTypeExistingCustomers= spark.sql("SELECT * FROM DuplicatePaymentFlags_Existing")
    DuplicatePaymentTypeExistingCustomers.createOrReplaceTempView("DuplicatePaymentTypeExistingCustomers")



### New Customers min/max date and duplicate Payment Types calculation

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    
    #New Customers
    calculate_min_max_paydate(NewCustomers,"FinalCalculationFieldsNew", spark)
    FinalNewCustomers= spark.sql("SELECT * FROM FinalCalculationFieldsNew")
    FinalNewCustomers.createOrReplaceTempView("FinalNewCustomers")

    find_duplicate_payment_type(FinalNewCustomers, AMLYear, DateTo,"DuplicatePaymentFlags_New", spark)
    DuplicatePaymentTypeNewCustomers= spark.sql("SELECT * FROM DuplicatePaymentFlags_New")
    DuplicatePaymentTypeNewCustomers.createOrReplaceTempView("DuplicatePaymentTypeNewCustomers")



### Create final tables

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    # All Duplicates 
    DuplicatePaymentTypeAllCustomers= spark.sql("SELECT * FROM DuplicatePaymentTypeNewCustomers UNION  SELECT * FROM DuplicatePaymentTypeExistingCustomers")
    DuplicatePaymentTypeAllCustomers.createOrReplaceTempView("DuplicatePaymentTypeAllCustomers")

    ## Include all existing and non existing customers
    OverallPayments = spark.sql(f"""
                                SELECT Ruleid, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, PortfolioName
                                FROM  FinalNewCustomers -- New customers without any prior payments
                                
                                UNION
        
                                SELECT Ruleid, b.TaxNumber, b.PaymentAmount, NumOfPayment, NumOfPortfolio, c.PortfolioName
                                FROM FinalExistingCustomers b
                                join amlPerimeter c on b.taxnumber = c.TaxNumber
                                where c.PortfolioName in (select PortfolioName from PortfolioPerimeter)  -- New payments - Existing customers
        
                                """)

    OverallPayments.createOrReplaceTempView("OverallPayments")

    FinalRule23 = spark.sql(f"""
                                        SELECT DISTINCT  RuleId
                                                , a.TaxNumber
                                                , a.PaymentAmount
                                                , NumOfPayment
                                                , NumOfPortfolio
                                                , CASE WHEN c.TaxNumber IS NOT NULL THEN NULL ELSE b.PaymentType END AS PaymentType
                                                , a.PortfolioName
                                        FROM OverallPayments a
                                        INNER JOIN amlPerimeter	b
                                        ON a.TaxNumber = b.TaxNumber
                                        LEFT JOIN DuplicatePaymentTypeAllCustomers c
                                        ON c.TaxNumber = a.TaxNumber 
                                        WHERE year(b.PayDateRd) =	year(getdate())
        
                                                            """)
    FinalRule23.createOrReplaceTempView("FinalRule23")


In [0]:
if AMLExecutionFlag == 1:
	FinalTableRank = spark.sql(f"""
							select	distinct cast(getdate() as date) Report_Date
									,case when n.taxnumber is not null then n.minPayDate ELSE DateFrom END  DateFrom
									,case when n.taxnumber is not null then n.maxPayDate ELSE DateTo END DateTo	
									,a.exposurekey as AccountNumber
									,a.CDI CDI
									,a.taxnumber Tax_Number
									,ltrim(rtrim(concat(a.firstname, ' ', a.lastname, ' ', a.fathername))) Customer_Name
									,a.legalentity Legal_Entity
									,a.Nationality
									,CASE WHEN a.legalentity = 'No' THEN  cast(a.birthdate as date) ELSE null END Birth_Date
									,a.Occupation 
									,f.PortfolioName portfolio_of_payment
									,f.PaymentAmount payment_amount
									,substring(f.paymenttype,1,199) PaymentType
									,f.numofpayment payments
									,CASE WHEN comm.TaxNumber is not null THEN comm.origin ELSE f.PortfolioName END Customer_Portfolio
									,f.ruleid
									,'M' AS rule_type
									,'YtD sum of installments/ payment >=15.000 per unique TAX number across all NOT BANKING portfolios (not per λογαριασμό)' AS rule_name
									,amlcheck.aml_sch_id amlSchIdexe
									,ROW_NUMBER ( )   
								OVER (partition by a.taxnumber, f.paymentamount order by a.CDI ) rnk
							from FinalRule23 f 
								join amlPerimeter a on a.TaxNumber = f.TaxNumber	
								join PortfolioPerimeter p on left(a.cdi,3) = p.Prefix
								join AmlExecutionCalendar amlcheck on PayDateRd between AMLYear and DateTo
								left join amlCommonCustomers comm on comm.TaxNumber=f.TaxNumber
								left join FinalNewCustomers n on n.taxnumber = a.taxnumber
	""")


	FinalTableRank_withchecksum = FinalTableRank.withColumn(
		"checksum",
		crc32(concat_ws("|", "cdi", "amlSchIdexe", "ruleid"))
	)
	FinalTableRank_withchecksum.createOrReplaceTempView("FinalTableRank_withchecksum")


In [0]:
if AMLExecutionFlag == 1:
	amlReportRun = spark.sql(f"""
							SELECT DISTINCT NULL as id
       										, Report_Date 
											, DateFrom date_from
											, DateTo date_to
											, CDI
											, Tax_Number
											,Customer_Name
											, Legal_Entity
											, Nationality
											, Birth_Date
											, Occupation 
											, portfolio_of_payment portfolio_payment
											, payment_amount
											, payments num_of_payments
											, Customer_Portfolio 
											, ruleid rule_id
											, rule_type
											, rule_name
											, amlSchIdexe aml_sch_id
											, checksum      chsum   
											, accountnumber account_number
											, PaymentType payment_type


							FROM FinalTableRank_withchecksum
							WHERE rnk = 1""") 

	amlReportRun.createOrReplaceTempView("amlReportRun")


### Create index for final table (each row has an incremental id)

In [0]:
if AMLExecutionFlag ==1:
    # Get the max ID from the table (default to 0 if empty)
    max_id = aml_report_view.agg({"id": "max"}).first()[0] or 0

    # Add incremental ID to view starting from max_id + 1 e.g. if max_id = 1692 then id = 1692 +1 then +2 etc.
    amlReportRun_withid = amlReportRun.withColumn(
        "id", monotonically_increasing_id() + (max_id + 1)
    )

    amlReportRun_withid.createOrReplaceTempView("amlReportRun_withid")



In [0]:
if AMLExecutionFlag ==1:
       update_portfolio_of_payment(DateFrom, DateTo , spark)
       final_df = spark.sql("""
        SELECT t.*, 
               COALESCE(t.portfolio_payment, s.portfolio) AS updated_portfolio
        FROM amlReportRun_withid t
        LEFT JOIN CustomerPortfolioAggtemp s
        ON t.Tax_number = s.taxnumber
       """).drop("portfolio_payment") \
      .withColumnRenamed("updated_portfolio", "portfolio_payment")
    
       final_df.createOrReplaceTempView("amlReportRun_withid")


In [0]:
if AMLExecutionFlag == 1:
  
	final = spark.sql(f""" SELECT               id
                  							, Report_Date
                                            , date_from
                                            , date_to
                                            , CDI
											, Tax_Number
											, Customer_Name
											, Legal_Entity
											, Nationality
											, Birth_Date
											, Occupation 
											, portfolio_payment
											, payment_amount
											, num_of_payments
											, Customer_Portfolio 
											, rule_id
											, rule_type
											, rule_name
											, aml_sch_id
											, chsum   
											, account_number
											, payment_type
                            FROM aml_report_view
                            UNION ALL
                            SELECT id
                            				,Report_Date
                                            , date_from
                                            , date_to
                                            , CDI
											, Tax_Number
											, Customer_Name
											, Legal_Entity
											, Nationality
											, Birth_Date
											, Occupation 
											, portfolio_payment
											, payment_amount
											, num_of_payments
											, Customer_Portfolio 
											, rule_id
											, rule_type
											, rule_name
											, aml_sch_id
											, chsum   
											, account_number
											, payment_type
                            FROM amlReportRun_withid
	""")
	final.createOrReplaceTempView("final")


## Append to final Calendar and AML Tables

In [0]:

if AMLExecutionFlag == 1:
  final.write.mode("overwrite").saveAsTable(f"`{env}-cdh`.delivery.aml_report") 
  CurrentExecution = spark.sql(f""" 
                                        SELECT d.id, execyear,exectype,execpart,datefrom,dateTo,executiondate, getdate() actualexecution,execmode     
                                        FROM execution_calendar d
                                        WHERE  month(d.executiondate) = month(getdate())
                                        AND year(d.executiondate) = year(getdate())
                                        """)
  CurrentExecution.createOrReplaceTempView("CurrentExecution")
  OtherExecutions = spark.sql(f"""  SELECT d.*    
                                      FROM execution_calendar d
                                      join CurrentExecution a on d.datefrom <> a.datefrom
                                    
                                        """)
  OtherExecutions.createOrReplaceTempView("OtherExecutions")

  Updated_execution_calendar = spark.sql(f"""  SELECT *   
                                      FROM CurrentExecution
                                      UNION 
                                      SELECT * 
                                      FROM OtherExecutions
                                        """)
  Updated_execution_calendar.createOrReplaceTempView("Updated_execution_calendar")

  Updated_execution_calendar.write.mode("overwrite").saveAsTable(f"`{env}-cdh`.delivery.execution_calendar")  


